In [1]:
%pip install -q datasets tokenizers pyyaml numpy

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
paddlex 3.7.2 requires numpy<2.4,>=1.24, but you have numpy 2.4.6 which is incompatible.
paddlex 3.7.2 requires PyYAML==6.0.2, but you have pyyaml 6.0.3 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import re, math, json, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

WORKDIR = Path("uzbek_llm_run")
WORKDIR.mkdir(exist_ok=True)

Device: cuda
GPU: NVIDIA GeForce RTX 3060


In [3]:
# 01. Hugging Face dan ikkita uzbek datasetini olish
from huggingface_hub import hf_hub_download

MAX_DOCS = 10000
MAX_CHARS = 4000

texts, seen = [], set()

# Matnlarni tozalash va qushish
def add_text(text):
    if not isinstance(text, str):
        return

    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    bad_phrases = [
        "creative commons",
        "litsenziyasi",
        "all rights reserved",
        "copyright",
        "metadata"
    ]

    if len(text) < 100:
        return

    if any(p in text.casefold() for p in bad_phrases):
        return

    text = text[:MAX_CHARS]
    key = text.casefold()

    if key not in seen:
        seen.add(key)
        texts.append(text)


# 1. xkas2001 datasetini yuklash
print("1. xkas2001 yuklanmoqda...")

file_path = hf_hub_download(
    repo_id="xkas2001/uzbek-language-dataset",
    repo_type="dataset",
    filename="community-oscar-uzbek/top5_metadata_text.txt"
)

with open(file_path, "r", encoding="utf-8") as f:
    for line in tqdm(f):
        add_text(line)

        if len(texts) >= MAX_DOCS:
            break

xkas_count = len(texts)
print("xkas2001:", xkas_count)


# 2. Digitaluz datasetini yuklash
print("\n2. Digitaluz yuklanmoqda...")

stream = load_dataset(
    "Digitaluz/uzbek-language-corpus",
    split="train",
    streaming=True
)

digitaluz_count = 0

for row in tqdm(stream):
    add_text(row.get("text", ""))

    if len(texts) >= xkas_count + MAX_DOCS:
        break

digitaluz_count = len(texts) - xkas_count

# Natijalar
print("\n--- Dataset natijalari ---")
print("xkas2001:", xkas_count)
print("Digitaluz:", digitaluz_count)
print("Jami unikal hujjatlar:", len(texts))

assert len(texts) >= 100, "Dataset dan yetarli matn olinmadi."

# example larni kurish
for i, text in enumerate(texts[:5], 1):
    print(f"\n--- Namuna {i} ---")
    print(text[:500])

1. xkas2001 yuklanmoqda...


community-oscar-uzbek/top5_metadata_text(…): reconstructing file:   0%|          |  0.00B /  167MB            

community-oscar-uzbek/top5_metadata_text(…): downloading bytes:           |  0.00B            

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\datasets--xkas2001--uzbek-language-dataset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


0it [00:00, ?it/s]

xkas2001: 10000

2. Digitaluz yuklanmoqda...


... (2 outputs left out)

Resolving data files:   0%|          | 0/136 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/136 [00:00<?, ?it/s]

0it [00:00, ?it/s]


--- Dataset natijalari ---
xkas2001: 10000
Digitaluz: 10000
Jami unikal hujjatlar: 20000

--- Namuna 1 ---
Ogʻiz boʻshligʻi — hazm yoʻlining boshlangʻich boʻlimi. Old va yon tomonlari lunj bilan, yuqoridan yumshoq va qattiq tanglay, pastdan Ogʻiz boʻshligʻi tubi — til osti muskullari bilan chegaralangan. Ogʻiz boʻshligʻi orqa tomondan toq teshik—tomoq orqali halqum boʻshligʻiga tutashadi. Tishlar va milklar Ogʻiz boʻshligʻini ogʻiz dahlizi (ogʻizga kiraverish) bilan asli Ogʻiz boʻshligʻiga ajratib turadi. Ogʻiz daxlizi ogʻiz teshigi orqali tashqi muhitga tutashadi. Ogʻiz yumilganda Ogʻiz boʻshligʻi t

--- Namuna 2 ---
Darvoqe, kuzning ham oʻz bahori bor. Kechagina xunuk koʻrinib turgan yulgʻunlarning pushti koʻylagi sayhonlik, dalayu yoʻl chetlariga goʻyo koʻklam keltirgandek.

--- Namuna 3 ---
— Onajon, qarang chiroyliligini! Kuzda ham daraxt gullabdi. Oʻqituvchimiz daraxtlar faqat bahorda gullaydi, degandilar-ku...

--- Namuna 4 ---
— Oʻgʻlim, bu daraxt emas, buta, uni yulgʻun deyis

In [4]:
# Takroriy matnlarni tekshirish
print("Jami:", len(texts))
print("Unikal:", len(set(texts)))

Jami: 20000
Unikal: 20000


In [5]:
# 02. Train/validation ajratish va BPE tokenizer yasash
WORKDIR = Path("uzbek_llm_run")
WORKDIR.mkdir(parents=True, exist_ok=True)

random.shuffle(texts)
split_at = max(1, int(len(texts) * 0.98))
train_texts, val_texts = texts[:split_at], texts[split_at:]

tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=8000,
    min_frequency=2,
    special_tokens=["[UNK]", "[BOS]", "[EOS]"]
)
tokenizer.train_from_iterator(train_texts, trainer=trainer)
TOKENIZER_PATH = WORKDIR / "uzbek_tokenizer.json"
tokenizer.save(str(TOKENIZER_PATH))

vocab = tokenizer.get_vocab()
EOS_ID = vocab["[EOS]"]
print("Vocabulary:", tokenizer.get_vocab_size())
print("Tokenizer:", TOKENIZER_PATH)

Vocabulary: 8000
Tokenizer: uzbek_llm_run\uzbek_tokenizer.json


In [6]:
# 03. Matnlarni token ID larga aylantirish
def encode_documents(documents):
    ids = []
    for text in tqdm(documents):
        ids.extend(tokenizer.encode(text).ids)
        ids.append(EOS_ID)
    return np.asarray(ids, dtype=np.uint16)

train_ids = encode_documents(train_texts)
val_ids = encode_documents(val_texts)

TRAIN_PATH = WORKDIR / "train.bin"
VAL_PATH = WORKDIR / "validation.bin"
train_ids.tofile(TRAIN_PATH)
val_ids.tofile(VAL_PATH)

print("Train tokenlari:", len(train_ids))
print("Validation tokenlari:", len(val_ids))
assert len(train_ids) > 1000 and len(val_ids) > 100

  0%|          | 0/19600 [00:00<?, ?it/s]

  0%|          | 0/400 [00:00<?, ?it/s]

Train tokenlari: 5286975
Validation tokenlari: 95572


In [7]:
# 04. Kichik GPT (decoder-only Transformer) modelini yasash
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, block_size=128, n_embd=256, n_head=4, n_layer=4, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        layer = nn.TransformerEncoderLayer(
            d_model=n_embd, nhead=n_head, dim_feedforward=4*n_embd,
            dropout=dropout, batch_first=True, norm_first=True, activation="gelu"
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=n_layer)
        self.norm = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

    def forward(self, idx, targets=None):
        batch, seq_len = idx.shape
        if seq_len > self.block_size:
            raise ValueError("Sequence block_size'dan uzun.")
        pos = torch.arange(seq_len, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(pos)[None, :, :]
        # Causal mask: model kelajakdagi tokenlarni kura olmaydi.
        mask = torch.triu(torch.ones(seq_len, seq_len, device=idx.device, dtype=torch.bool), diagonal=1)
        x = self.blocks(x, mask=mask)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=80, temperature=0.8, top_k=40):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k:
                values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < values[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx

VOCAB_SIZE = tokenizer.get_vocab_size()
BLOCK_SIZE = 128
model = GPTLanguageModel(VOCAB_SIZE, block_size=BLOCK_SIZE).to(DEVICE)
print("Parametrlar:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Model device:", DEVICE)

Parametrlar: 5,240,320
Model device: cuda


C:\Users\user\AppData\Local\Temp\ipykernel_9596\1587760457.py:12: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(layer, num_layers=n_layer)


In [8]:
# 05. Batch olish va train/validation loss hisoblash
train_data = torch.from_numpy(train_ids.astype(np.int64))
val_data = torch.from_numpy(val_ids.astype(np.int64))

BATCH_SIZE = 4 if DEVICE == "cuda" else 2
def get_batch(data):
    ix = torch.randint(len(data) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([data[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([data[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

@torch.no_grad()
def estimate_loss(eval_iters=20):
    model.eval()
    result = {}
    for name, data in [("train", train_data), ("val", val_data)]:
        losses = []
        for _ in range(eval_iters):
            x, y = get_batch(data)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE=="cuda")):
                _, loss = model(x, y)
            losses.append(loss.item())
        result[name] = float(np.mean(losses))
    model.train()
    return result

print("Start loss:", estimate_loss(eval_iters=5))

Start loss: {'train': 153.207421875, 'val': 153.5578125}


In [9]:
# 06. Pretraining loop: AdamW, gradient clipping va learning rate
MAX_STEPS = 1000       # Dastlabki tajriba; sifatli model uchun ko'proq step kerak
LEARNING_RATE = 3e-4
EVAL_INTERVAL = 100
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1)
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

history = []
best_val_loss = float("inf")
CHECKPOINT_PATH = WORKDIR / "best_model.pt"

for step in tqdm(range(1, MAX_STEPS + 1)):
    model.train()
    x, y = get_batch(train_data)
    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
        _, loss = model(x, y)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    if step % EVAL_INTERVAL == 0 or step == 1:
        metrics = estimate_loss()
        history.append({"step": step, **metrics})
        print(f"step {step}: train={metrics['train']:.4f}, val={metrics['val']:.4f}")

        if metrics["val"] < best_val_loss:
            best_val_loss = metrics["val"]
            torch.save({
                "step": step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": best_val_loss,
                "config": {"vocab_size": VOCAB_SIZE, "block_size": BLOCK_SIZE}
            }, CHECKPOINT_PATH)

with open(WORKDIR / "training_log.json", "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

print("Best validation loss:", best_val_loss)
print("Saved checkpoint:", CHECKPOINT_PATH)

  0%|          | 0/1000 [00:00<?, ?it/s]

step 1: train=147.4187, val=147.5368
step 100: train=23.3304, val=23.4411
step 200: train=17.2373, val=17.2641
step 300: train=13.8557, val=13.6749
step 400: train=11.9830, val=11.7158
step 500: train=10.7204, val=10.7366
step 600: train=9.9906, val=9.9488
step 700: train=9.3877, val=9.3366
step 800: train=9.1418, val=8.8944
step 900: train=8.7691, val=8.9391
step 1000: train=8.5778, val=8.5067
Best validation loss: 8.506718635559082
Saved checkpoint: uzbek_llm_run\best_model.pt


In [10]:
# 07. Eng yaxshi checkpointni yuklash va perplexity hisoblash
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

final_metrics = estimate_loss(eval_iters=30)
val_perplexity = math.exp(min(final_metrics["val"], 20))
print("Train loss:", round(final_metrics["train"], 4))
print("Validation loss:", round(final_metrics["val"], 4))
print("Validation perplexity:", round(val_perplexity, 2))

Train loss: 8.4858
Validation loss: 8.5389
Validation perplexity: 5109.53


In [11]:
# 08. Uzbekcha matn generation & save model
prompt = "O'zbekistonning tabiati"
prompt_ids = tokenizer.encode(prompt).ids
input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)

generated_ids = model.generate(input_ids, max_new_tokens=80)[0].tolist()
print(tokenizer.decode(generated_ids))

# Model va tokenizer alohida saqlanadi.
torch.save(model.state_dict(), WORKDIR / "uzbek_gpt_weights.pt")
print("Natijalar papkasi:", WORKDIR.resolve())
print("Checkpoint:", CHECKPOINT_PATH)
print("Tokenizer:", TOKENIZER_PATH)

O'zbekistonning tabiati Oʻzbekiston ( “ “si, bu Oʻzbekiston. ". bu. –sh.: va – – (, (dan, ham ham,.da- –.. «dada. O O. O boʻlibgadada. ( “ O, bir ( bu:. “ “ “ “ – –, “ “ “ “ “ «, uda, (---
Natijalar papkasi: C:\uzbek_pretraining\uzbek_llm_run
Checkpoint: uzbek_llm_run\best_model.pt
Tokenizer: uzbek_llm_run\uzbek_tokenizer.json


In [12]:
# 09. Modelga savol berish va javob olish

def ask_model(question, max_new_tokens=100):
    model.eval()

    prompt_ids = tokenizer.encode(question).ids
    input_ids = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=DEVICE
    )

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_k=40
        )

    answer = tokenizer.decode(output_ids[0].tolist())

    print("\nSavol:", question)
    print("Javob:", answer[len(question):].strip())

question = "O'zbekistonning poytaxti qaysi shahar?"
ask_model(question)


Savol: O'zbekistonning poytaxti qaysi shahar?
Javob: ””- bir va.sida,. bu.. “ “ “da. va “ “ (., “, u. men. O.- ulari.da.mi, bu,-. Oʻzbekiston.,lar. ". uningdagada “ “ (. G. I I”,.,-dachacha.. –.... “. uningsi. bu., deb-da?
